# 📚 NB01：RAG 基礎概念與完整流程

**目標：** 理解什麼是 RAG、為什麼需要它，並從零開始實作一個完整的 RAG 系統。

**學習成果：**
- 理解 RAG 的核心概念與動機
- 掌握 RAG 完整五步驟流程
- 能用 ChromaDB + OpenAI 實作基本 RAG
- 了解 RAG vs Fine-tuning 的適用場景差異

## 1. 什麼是 RAG？

**RAG（Retrieval-Augmented Generation，檢索增強生成）** 是一種讓 LLM 在回答問題前，先從外部知識庫中「查詢相關資料」的技術架構。

### 核心概念類比

想像 LLM 是一位非常博學的顧問，但他的知識截止於訓練日期。
- ❌ **沒有 RAG：** 顧問全憑記憶回答，可能記錯或回答過時資訊（幻覺 Hallucination）
- ✅ **有 RAG：** 顧問先查閱相關文件再回答，答案更準確且有依據

### RAG 解決的核心問題

| 問題 | 說明 |
|------|------|
| **幻覺（Hallucination）** | LLM 捏造不存在的資訊 |
| **知識截止（Knowledge Cutoff）** | 模型無法知道訓練後發生的事 |
| **私有資料無法訓練** | 公司內部文件、機密資料不適合放入訓練集 |
| **知識更新成本高** | Fine-tuning 需要重新訓練，昂貴且緩慢 |

## 2. RAG 完整五步驟流程

```
┌─────────────────────────────────────────────────────────────┐
│                    RAG 完整流程                              │
│                                                             │
│  📄 文件                                                    │
│   ↓                                                         │
│  ✂️  Step 1: Chunking（切塊）                               │
│   ↓                                                         │
│  🔢 Step 2: Embedding（嵌入向量）                           │
│   ↓                                                         │
│  🗄️  Step 3: Vector Store（向量資料庫儲存）                  │
│                                                             │
│  👤 使用者提問                                              │
│   ↓                                                         │
│  🔍 Step 4: Retrieval（檢索相關 chunks）                    │
│   ↓                                                         │
│  🤖 Step 5: Generation（LLM 根據 context 生成答案）         │
│   ↓                                                         │
│  💬 回答                                                    │
└─────────────────────────────────────────────────────────────┘
```

前三步是**離線索引（Offline Indexing）**，後兩步是**線上查詢（Online Query）**。

## 3. 環境設定

In [ ]:
# 安裝相依套件（使用 uv）
# 在 terminal 執行：
# uv sync
# uv run jupyter notebook

# 或在 notebook 內直接安裝：
# %pip install openai chromadb python-dotenv tiktoken

In [ ]:
import os
import json
import textwrap
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

# 載入 .env 檔案中的 API Key
load_dotenv()

# 初始化 OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 初始化 ChromaDB（in-memory，不需要伺服器）
chroma_client = chromadb.Client()

print("✅ 環境設定完成")
print(f"OpenAI API Key 存在: {bool(os.getenv('OPENAI_API_KEY'))}")

## 4. 準備示範文件

我們使用一批關於 AI 公司和產品的文件作為知識庫。
這些資訊模擬「公司內部知識庫」或「最新新聞資料」。

In [ ]:
# 模擬知識庫文件（實際場景可能來自 PDF、網頁、資料庫等）
documents = [
    {
        "id": "doc_001",
        "title": "OpenAI GPT-4o 介紹",
        "content": "GPT-4o 是 OpenAI 於 2024 年 5 月發布的多模態大型語言模型。"
                   "它能夠同時處理文字、圖片和音訊輸入，並生成文字和音訊輸出。"
                   "GPT-4o 相較於前代模型速度提升了 2 倍，成本降低了 50%。"
                   "在推理、數學和程式設計能力上表現優異，是目前最強大的 OpenAI 商用模型之一。"
    },
    {
        "id": "doc_002",
        "title": "Anthropic Claude 系列介紹",
        "content": "Claude 是 Anthropic 公司開發的 AI 助理系列。"
                   "Claude 3 系列包含 Haiku、Sonnet 和 Opus 三個版本，分別針對速度、平衡和最高能力設計。"
                   "Anthropic 特別強調 AI 安全性，使用憲法 AI（Constitutional AI）訓練方法，"
                   "讓模型更加無害且誠實。Claude 在長文本理解和程式碼生成上表現突出。"
    },
    {
        "id": "doc_003",
        "title": "Google Gemini 介紹",
        "content": "Gemini 是 Google DeepMind 開發的多模態 AI 模型系列。"
                   "Gemini 1.5 Pro 具備高達 100 萬 token 的超長上下文窗口，"
                   "能夠處理整部書籍或一小時的影片。"
                   "Gemini 與 Google 生態系統深度整合，包括 Google Search、Gmail 和 Google Docs。"
                   "在多語言理解上表現優秀，特別支援繁體中文任務。"
    },
    {
        "id": "doc_004",
        "title": "向量資料庫技術比較",
        "content": "向量資料庫（Vector Database）是 RAG 系統的核心元件，用於儲存和檢索嵌入向量。"
                   "主要選項包括：Pinecone（雲端託管、易擴展）、Weaviate（開源、支援混合搜索）、"
                   "Qdrant（高效能、Rust 實作）、ChromaDB（輕量、適合原型開發）。"
                   "選擇向量資料庫時需考慮延遲需求、資料規模、成本和部署方式。"
    },
    {
        "id": "doc_005",
        "title": "RAG 系統架構最佳實踐",
        "content": "建構 RAG 系統時有幾個關鍵最佳實踐："
                   "1. Chunk Size 選擇：通常 200-500 token 效果較好，過大會引入雜訊，過小會失去上下文。"
                   "2. 使用 Overlap：相鄰 chunk 之間保留 10-20% 重疊，避免語義被截斷。"
                   "3. Metadata 過濾：儲存文件來源、日期等元資料，用於縮小搜索範圍。"
                   "4. 混合搜索：結合語義搜索和關鍵字搜索，提升召回率。"
                   "5. Reranking：對初步檢索結果二次排序，提升精確度。"
    },
    {
        "id": "doc_006",
        "title": "台灣 AI 新創生態系",
        "content": "台灣在 AI 領域有活躍的新創生態系，特別是在硬體和 AI 應用層面。"
                   "NVIDIA 在台灣設有研發中心，台積電是全球最重要的 AI 晶片代工廠。"
                   "本土 AI 新創如 Appier（沛星互動科技）已在東京上市，"
                   "專注於行銷 AI 解決方案。"
                   "政府推動 AI Taiwan 計畫，投入資源培育 AI 人才和產業應用。"
    }
]

print(f"已準備 {len(documents)} 份文件")
for doc in documents:
    print(f"  - [{doc['id']}] {doc['title']}")

## Step 1 + 2：切塊（Chunking）+ 嵌入向量（Embedding）

### 什麼是 Embedding？

Embedding（嵌入向量）是將文字轉換為**高維度數值向量**的技術。
語意相近的文字，在向量空間中的距離也相近。

```
"台灣的 AI 發展" → [0.23, -0.45, 0.89, ..., 0.12]  (1536 維)
"人工智慧在台灣" → [0.25, -0.43, 0.87, ..., 0.11]  (相似！)
"今天天氣如何"  → [-0.89, 0.32, -0.21, ..., 0.67] (差異大！)
```

### 本例中的簡化：
- 每份文件直接作為一個 chunk（NB02 會深入探討切塊策略）
- 使用 OpenAI `text-embedding-3-small` 模型生成向量

In [ ]:
def create_embedding(text: str) -> list[float]:
    """使用 OpenAI API 將文字轉為嵌入向量"""
    response = client.embeddings.create(
        model="text-embedding-3-small",  # 1536 維，費用低，品質好
        input=text
    )
    return response.data[0].embedding

# 示範：看看 embedding 長什麼樣子
sample_text = "什麼是 RAG？"
sample_embedding = create_embedding(sample_text)

print(f"文字：'{sample_text}'")
print(f"向量維度：{len(sample_embedding)}")
print(f"前 10 個值：{[round(v, 4) for v in sample_embedding[:10]]}")
print(f"\n→ 這個向量就是文字的數學表示，語意相近的文字會有相近的向量！")

## Step 3：向量資料庫（Vector Store）

將所有文件的嵌入向量存入 ChromaDB，建立可搜索的向量索引。

In [ ]:
# 建立 ChromaDB collection（類似資料庫中的 table）
# 如果已存在就刪除重建
try:
    chroma_client.delete_collection("rag_demo_nb01")
except:
    pass

collection = chroma_client.create_collection(
    name="rag_demo_nb01",
    metadata={"hnsw:space": "cosine"}  # 使用餘弦相似度
)

print("開始索引文件...")
print("-" * 50)

for doc in documents:
    # 生成嵌入向量
    embedding = create_embedding(doc["content"])
    
    # 存入 ChromaDB
    collection.add(
        ids=[doc["id"]],
        embeddings=[embedding],
        documents=[doc["content"]],
        metadatas=[{"title": doc["title"]}]
    )
    print(f"✅ 已索引：{doc['title']}")

print("-" * 50)
print(f"\n向量資料庫共有 {collection.count()} 份文件")

## Step 4：檢索（Retrieval）

當使用者提問時，將問題也轉換為向量，然後在向量資料庫中找出**最相似**的文件。

相似度計算使用**餘弦相似度（Cosine Similarity）**：
$$\text{similarity}(A, B) = \frac{A \cdot B}{|A| \cdot |B|}$$

值越接近 1，代表越相似。

In [ ]:
def retrieve(query: str, n_results: int = 3) -> dict:
    """從向量資料庫檢索最相關的文件"""
    # 將問題轉為向量
    query_embedding = create_embedding(query)
    
    # 在向量資料庫中搜索
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    return results

# 測試檢索
test_query = "哪個 AI 模型支援最長的上下文窗口？"
results = retrieve(test_query, n_results=3)

print(f"查詢：'{test_query}'")
print("=" * 60)
print("檢索到的相關文件：\n")

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    similarity = 1 - dist  # ChromaDB 回傳的是距離，轉換為相似度
    print(f"[排名 {i+1}] {meta['title']}")
    print(f"相似度：{similarity:.4f}")
    print(f"內容：{doc[:100]}...")
    print()

## Step 5：生成（Generation）

將檢索到的文件作為 **Context（背景資料）** 注入 Prompt，讓 LLM 根據這些資料回答問題。

### Prompt 結構
```
[System Prompt]
你是一個知識問答助理。請根據以下提供的「參考資料」回答問題。
如果參考資料中沒有相關資訊，請說明你不知道，不要捏造答案。

參考資料：
--- 文件 1 ---
{retrieved_doc_1}
--- 文件 2 ---
{retrieved_doc_2}

[User Message]
{user_question}
```

In [ ]:
def generate_answer(query: str, context_docs: list[str], context_titles: list[str]) -> str:
    """根據 context 文件生成回答"""
    # 組合 context
    context_parts = []
    for i, (title, doc) in enumerate(zip(context_titles, context_docs)):
        context_parts.append(f"--- 參考資料 {i+1}：{title} ---\n{doc}")
    context = "\n\n".join(context_parts)
    
    # 建構 Prompt
    system_prompt = """你是一個專業的知識問答助理。
請根據以下「參考資料」回答使用者的問題。
- 只使用參考資料中的資訊
- 如果參考資料沒有相關資訊，請誠實說明
- 回答要清晰簡潔，使用繁體中文"""
    
    user_message = f"""參考資料：
{context}

問題：{query}"""
    
    # 呼叫 LLM 生成回答
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0.1  # 降低隨機性，讓回答更穩定
    )
    
    return response.choices[0].message.content

# 測試生成
answer = generate_answer(
    query=test_query,
    context_docs=results["documents"][0],
    context_titles=[m["title"] for m in results["metadatas"][0]]
)

print(f"問題：{test_query}")
print("=" * 60)
print(f"RAG 回答：\n{answer}")

## 完整 RAG Pipeline 函式

將上面的步驟整合成一個完整的 RAG 函式。

In [ ]:
def rag_query(query: str, n_results: int = 3, verbose: bool = True) -> str:
    """完整的 RAG 查詢流程"""
    if verbose:
        print(f"🔍 查詢：{query}")
        print("-" * 50)
    
    # Step 4: 檢索
    results = retrieve(query, n_results)
    docs = results["documents"][0]
    titles = [m["title"] for m in results["metadatas"][0]]
    distances = results["distances"][0]
    
    if verbose:
        print("📚 檢索到的相關文件：")
        for title, dist in zip(titles, distances):
            similarity = 1 - dist
            print(f"  • {title} (相似度: {similarity:.3f})")
        print("-" * 50)
    
    # Step 5: 生成
    answer = generate_answer(query, docs, titles)
    
    if verbose:
        print("💬 RAG 回答：")
        print(answer)
    
    return answer

print("RAG 函式已定義 ✅")

In [ ]:
# 🧪 測試 1：AI 模型相關問題
print("=" * 60)
_ = rag_query("GPT-4o 和 Claude 有什麼不同？")

In [ ]:
# 🧪 測試 2：RAG 架構相關問題
print("=" * 60)
_ = rag_query("建立 RAG 系統時，chunk size 要怎麼選擇？")

In [ ]:
# 🧪 測試 3：知識庫外的問題（測試幻覺防範）
print("=" * 60)
_ = rag_query("2024 年台灣的 GDP 是多少？")

## 對比實驗：有 RAG vs 沒有 RAG

讓我們看看同一個問題，有無 RAG 的回答差異。

In [ ]:
def answer_without_rag(query: str) -> str:
    """直接問 LLM，沒有檢索步驟"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "你是一個知識問答助理。請回答使用者的問題。"},
            {"role": "user", "content": query}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content

# 問一個我們知識庫有、但 LLM 可能不確定的問題
comparison_query = "Anthropic Claude 使用什麼特殊的訓練方法？"

print("問題：", comparison_query)
print("\n" + "=" * 60)
print("❌ 沒有 RAG 的回答（純 LLM）：")
print("-" * 60)
vanilla_answer = answer_without_rag(comparison_query)
print(vanilla_answer)

print("\n" + "=" * 60)
print("✅ 有 RAG 的回答（有參考資料）：")
print("-" * 60)
rag_answer = rag_query(comparison_query, verbose=False)
print(rag_answer)

## RAG vs Fine-tuning 比較

| 面向 | RAG | Fine-tuning |
|------|-----|-------------|
| **知識更新** | ✅ 即時更新（修改向量庫即可） | ❌ 需要重新訓練 |
| **成本** | ✅ 低（只需 API 費用） | ❌ 高（GPU 訓練成本） |
| **可解釋性** | ✅ 可以看到引用來源 | ❌ 難以追蹤答案來源 |
| **幻覺風險** | ✅ 較低（有文件依據） | ❌ 較高 |
| **回應格式** | ❌ 格式控制較弱 | ✅ 可精確控制輸出格式 |
| **專業術語** | ❌ 需靠 context 補充 | ✅ 可學習特定領域用語 |
| **適用場景** | 知識庫問答、文件搜索、最新資訊 | 特定格式輸出、風格模仿、領域分類 |

### 面試關鍵問答
> **面試官：** 什麼情況下你會選擇 RAG 而不是 Fine-tuning？
>
> **好答案：** 「當知識需要頻繁更新（如新聞、文件庫），或需要追蹤答案來源時，RAG 更合適。Fine-tuning 適合當我需要改變模型的行為風格，或讓模型學習特定領域的輸出格式。」

## 小結

本 Notebook 我們學習了：

1. ✅ **RAG 的核心概念**：讓 LLM 查資料再回答，減少幻覺
2. ✅ **完整五步驟流程**：文件準備 → 切塊 → 嵌入 → 向量儲存 → 檢索 → 生成
3. ✅ **用 ChromaDB + OpenAI 實作基本 RAG**
4. ✅ **驗證 RAG 的效果**：有 vs 沒有 RAG 的回答差異

### 下一步
- **NB02**：深入探討 Chunking 策略，不同切塊方式對檢索品質的影響
- **NB03**：進階 RAG 技術：混合搜索（BM25 + 向量）+ 重排序（Reranking）
- **NB04**：RAG 評估框架：如何量化 RAG 系統的好壞